In [17]:
import numpy as np
import re

In [54]:
textlines = []
keywords = {}
linknumber = 0
output = "output.md"
with open("../Markdowns/modern-learning.md", "r") as f:
    lines = f.readlines()
for line in lines:
    line = line.rstrip()
    if line.startswith('- **'):
        keyword = line.lstrip('- **').rstrip('**').lower().replace(" ", "-")
        if keyword in keywords:
            raise ValueError("Duplicate keyword found.")
        keywords[keyword]=[]
        textlines.append(f'<a id="{keyword}"></a>')
    textlines.append(line)
for lineposition,line in enumerate(textlines):
    if line.startswith('- **'):
        entry_title = line.lstrip('- **').rstrip('**')
    positions = [m.start() for m in re.finditer(r"__", line)]
    if len(positions) % 2 != 0:
        raise ValueError("Unmatched __ found.")
    if positions:
        pos = 0
        segments = []
        for i in range(0, len(positions), 2):
            segments.append(line[pos:positions[i]])
            segments.append(line[positions[i]+2:positions[i+1]])
            pos = positions[i+1] + 2
        segments.append(line[pos:])
        for i in range(1, len(segments), 2):  # Process only the segments between __
            s = segments[i].lower().replace(" ", "-")
            found = False
            for k in keywords:
                if s in k:
                    if found:
                        raise ValueError("Multiple matches found for keyword.")
                    found = True
                    resolved_keyword = k
                    segments[i] = f"<a id='link-{linknumber}'></a>[{s}](#{resolved_keyword})"
                    keywords[k].append([f"link-{linknumber}", s, entry_title])
                    linknumber += 1
        textlines[lineposition] = "".join(segments)

final = []
for line in textlines:
    final.append(line)
    if line.startswith('- **'):
        keyword = line.lstrip('- **').rstrip('**').lower().replace(" ", "-")
        if keywords[keyword]:
            final.append('\t- referenced by:')
            for link in keywords[keyword]:
                final.append(f'\t\t- [{link[1]}](#{link[0]}) in section "{link[2]}"')

with open(output, "w") as f:
    f.write("\n".join(final))